In [1]:
# %% 셀 1: 텔롭 + 채널명 임베딩 (수정)
import os, json, torch
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

POS_DIR = "./data/8_telop_position"
EMB_SAVE_PATH = "./data/8_text_embeddings.pt"
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
BATCH_EMB = 2048


def last_token_pool(last_hidden_states, attention_mask):
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device),
        sequence_lengths
    ]


# ── unique 텍스트 수집 ──
unique_texts = set()
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    unique_texts.add(channel)
    for fname in sorted(os.listdir(ch_dir)):
        if not fname.endswith(".json"):
            continue
        with open(os.path.join(ch_dir, fname), "r") as f:
            data = json.load(f)
        for inst in data.get("instances", []):
            unique_texts.add(inst["text"])

unique_texts = sorted(unique_texts)
print(f"📝 unique 텍스트: {len(unique_texts):,}개")

# ── 임베딩 계산 ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')
model_emb = AutoModel.from_pretrained(MODEL_NAME).half().cuda().eval()

text2emb = {}
for i in tqdm(range(0, len(unique_texts), BATCH_EMB), desc="임베딩"):
    batch_texts = unique_texts[i:i + BATCH_EMB]
    inputs = tokenizer(batch_texts, padding=True, truncation=True,
                       max_length=128, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_emb(**inputs)
        embs = last_token_pool(outputs.last_hidden_state, inputs['attention_mask'])
        embs = embs.float().cpu()

    for text, emb in zip(batch_texts, embs):
        text2emb[text] = emb

torch.save(text2emb, EMB_SAVE_PATH)
print(f"✅ 저장: {EMB_SAVE_PATH} ({len(text2emb):,}개 텍스트)")

del model_emb, tokenizer
torch.cuda.empty_cache()

/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📝 unique 텍스트: 1,448,729개


Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
임베딩: 100%|██████████| 708/708 [10:44<00:00,  1.10it/s]


✅ 저장: ./data/8_text_embeddings.pt (1,448,729개 텍스트)


In [2]:
# %% 셀 2: STT 텍스트 임베딩 추가 (수정)
import os, json, torch
import numpy as np
import polars as pl
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

STT_DIR = "./data/4_stt_results"
EMB_SAVE_PATH = "./data/8_text_embeddings.pt"
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
BATCH_EMB = 1024
FPS = 10


def last_token_pool(last_hidden_states, attention_mask):
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device),
        sequence_lengths
    ]


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({"text": cur_text.strip()})
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({"text": cur_text.strip()})
    return segments


# ── 기존 임베딩 로드 ──
text2emb = torch.load(EMB_SAVE_PATH, weights_only=True)
print(f"기존 임베딩: {len(text2emb):,}개")

# ── STT unique 텍스트 수집 ──
stt_texts = set()
for channel in tqdm(sorted(os.listdir(STT_DIR)), desc="STT 수집"):
    ch_dir = os.path.join(STT_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if not fname.endswith(".parquet"):
            continue
        try:
            df = pl.read_parquet(os.path.join(ch_dir, fname), glob=False)
            segs = stt_frames_to_segments(df)
            for seg in segs:
                if seg["text"]:
                    stt_texts.add(seg["text"])
        except:
            continue

new_texts = sorted(stt_texts - set(text2emb.keys()))
print(f"STT unique 텍스트: {len(stt_texts):,}개")
print(f"신규 (임베딩 필요): {len(new_texts):,}개")

# ── 임베딩 계산 ──
if new_texts:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')
    model_emb = AutoModel.from_pretrained(MODEL_NAME).half().cuda().eval()

    for i in tqdm(range(0, len(new_texts), BATCH_EMB), desc="STT 임베딩"):
        batch_texts = new_texts[i:i + BATCH_EMB]
        inputs = tokenizer(batch_texts, padding=True, truncation=True,
                           max_length=256, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model_emb(**inputs)
            embs = last_token_pool(outputs.last_hidden_state, inputs['attention_mask'])
            embs = embs.float().cpu()

        for text, emb in zip(batch_texts, embs):
            text2emb[text] = emb

    del model_emb, tokenizer
    torch.cuda.empty_cache()

    torch.save(text2emb, EMB_SAVE_PATH)
    print(f"✅ 저장: {EMB_SAVE_PATH} ({len(text2emb):,}개 텍스트)")
else:
    print("✅ 신규 임베딩 없음, 기존 파일 유지")

기존 임베딩: 1,448,729개


STT 수집: 100%|██████████| 664/664 [01:10<00:00,  9.37it/s]


STT unique 텍스트: 684,500개
신규 (임베딩 필요): 653,227개


STT 임베딩: 100%|██████████| 638/638 [08:41<00:00,  1.22it/s]


✅ 저장: ./data/8_text_embeddings.pt (2,101,956개 텍스트)
